# 03 Gold analysis

Build the four denormalized gold tables and read them as a template for
BI or SQL consumers.

Run this notebook from PyCharm using the IDE **Managed Jupyter server**:
`Project Settings > Python Interpreter > Jupyter`, let PyCharm start and
manage the server, and select the project `.venv` as the kernel. No separate
`jupyter notebook` process is required (see `notebooks/README.md`).

## Bootstrap

In [1]:
import os
import sys
from pathlib import Path

# Make the project importable no matter where the kernel starts.
_here = Path.cwd()
_root = _here.parent if _here.name == "notebooks" else _here
sys.path.insert(0, str(_root))
os.chdir(_root)

# Force SQLite so notebooks never touch PostgreSQL by accident.
os.environ["ENV"] = "DEV"
os.environ.setdefault("ETL_SQLITE_PATH", "data/warehouse/violence_staging.db")

import pandas as pd  # noqa: F401  (used by later cells)

from violence_etl.config.settings import get_settings

settings = get_settings()
print("env    :", settings.env)
print("sqlite :", settings.sqlite_path)
print("bronze :", settings.bronze_dir)

backend: sqlite
sqlite : data/warehouse/violence_staging.db
bronze : data/bronze


## Build gold (full refresh) from silver

In [2]:
from violence_etl.db.connection import make_engine
from violence_etl.transformations.gold import build_gold

engine = make_engine(settings=settings)
summary = build_gold(engine, export_csv=True, export_dir="data/gold")
summary

{'grand_total': 15840,
 'gold_top5_departments': 5,
 'gold_crimes_by_department': 32,
 'gold_crimes_by_calendar': 1953,
 'gold_crimes_by_electoral_period': 4,
 'elapsed_seconds': 0.123}

## a. Top 5 departments by total crimes

`rank` 1..5, ordered by `total_crimes DESC`, tie broken by `departamento ASC`.

In [3]:
pd.read_sql("select * from gold_top5_departments order by rank", engine)

,rank,cod_depto,departamento,total_crimes
0,1,05,ANTIOQUIA,1918
1,2,19,CAUCA,1866
2,3,54,NORTE DE SANTANDER,1564
3,4,81,ARAUCA,1308
4,5,52,NARIÑO,1178


## b. Total crimes by department (all departments)

In [4]:
pd.read_sql(
    "select * from gold_crimes_by_department order by total_crimes desc",
    engine,
)

,cod_depto,departamento,total_crimes
0,05,ANTIOQUIA,1918
1,19,CAUCA,1866
2,54,NORTE DE SANTANDER,1564
3,81,ARAUCA,1308
4,52,NARIÑO,1178
5,50,META,993
6,76,VALLE DEL CAUCA,978
7,41,HUILA,689
8,86,PUTUMAYO,657
9,27,CHOCO,631


## c. Total crimes by year, quarter, month and weekday

Grain: `year x quarter x month x weekday_num` (unique).

In [5]:
calendar = pd.read_sql(
    "select * from gold_crimes_by_calendar "
    "where year = (select max(year) from gold_crimes_by_calendar) "
    "order by quarter, month, weekday_num",
    engine,
)
calendar

,year,quarter,month,weekday_num,weekday_name,total_crimes
0,2026,1,1,1,Monday,13
1,2026,1,1,2,Tuesday,11
2,2026,1,1,3,Wednesday,17
3,2026,1,1,4,Thursday,24
4,2026,1,1,5,Friday,20
...,...,...,...,...,...,...
58,2026,3,9,3,Wednesday,15
59,2026,3,9,4,Thursday,14
60,2026,3,9,5,Friday,6
61,2026,3,9,6,Saturday,12


In [6]:
# Aggregate a compact view: crimes by weekday across the whole history.
pd.read_sql(
    "select weekday_num, weekday_name, sum(total_crimes) as crimes "
    "from gold_crimes_by_calendar group by weekday_num, weekday_name "
    "order by weekday_num",
    engine,
)

,weekday_num,weekday_name,crimes
0,1,Monday,2116
1,2,Tuesday,2304
2,3,Wednesday,2362
3,4,Thursday,2409
4,5,Friday,2421
5,6,Saturday,2211
6,7,Sunday,2017


## d. Total crimes by electoral period

`period_order` uses the presentation order post, intermediate, pre, electoral.
`pct_of_total` is the share of the grand total, rounded to 2 decimals and sums to 100.

In [7]:
periods = pd.read_sql(
    "select * from gold_crimes_by_electoral_period order by period_order", engine
)
periods

,electoral_period,period_order,year_min,year_max,total_crimes,pct_of_total
0,post-electoral,1,2003,2023,3647,23.02
1,intermediate,2,2004,2024,3880,24.49
2,pre-electoral,3,2005,2025,4237,26.75
3,electoral,4,2006,2026,4076,25.73


In [8]:
print("sum pct_of_total:", periods["pct_of_total"].sum())
print("grand total     :", periods["total_crimes"].sum())

sum pct_of_total: 99.99
grand total     : 15840


## Equivalent plain SQL

```sql
-- b. total crimes by department
SELECT cod_depto, departamento, SUM(cantidad) AS total_crimes
FROM silver_crimes GROUP BY cod_depto, departamento ORDER BY total_crimes DESC;

-- c. total crimes by year, quarter, month and weekday
SELECT year, quarter, month, weekday_num, weekday_name, SUM(cantidad) AS total_crimes
FROM silver_crimes
GROUP BY year, quarter, month, weekday_num, weekday_name;

-- d. total crimes by electoral period
SELECT electoral_period, SUM(cantidad) AS total_crimes
FROM silver_crimes GROUP BY electoral_period;
```